# Project 4. Tardigrades: from genestealers to space marines

Исследуем геном тихоходки Ramazzottius varieornatus (штамм YOKOZUNA-1), чтобы понять генетические механизмы ее экстремальной устойчивости к радиации и дегридатации. Тихоходки могут выживать в открытом космосе, переносить высокое давление и дозы радиации, смертельные для других животных. 

Наша цель — найти и аннотировать белки, которые могут участвовать в восстановлении ДНК.

### 1. Получение данных

In [ ]:
%%bash

# создаём директорию
mkdir data
cd data

# загружаем готовую сборку генома Ramazzottius varieornatus (штамм YOKOZUNA-1) в fasta (fna) формате из базы NCBI
wget ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/001/949/185/GCA_001949185.1_Rvar_4.0/GCA_001949185.1_Rvar_4.0_genomic.fna.gz

# и распаковываем
gunzip GCA_001949185.1_Rvar_4.0_genomic.fna.gz

## Опциональная часть (маскировка повторов, предсказание генов, обучение модели)

### 1. Маскировка повторов

В эукариотических геномах много повторяющихся последовательностей. Если их не замаскировать, аглоритмы предсказания генов (например, AUGUSTUS) могут ошибочно принимать ORF транспозонов за настоящие экзоны организма, аннотация сломается.

Тихоходка - малоизученный организм, стандартные библиотеки повторов не подойдут.

Используем RepeatModeler для предсказания повторов, потом скормим эту библиотеку RepeatMasker.

In [ ]:
# много читала форум biostars, чтобы понять какой пайплайн даст баланс между временем и качеством.
# надеюсь, получится неплохо...

In [ ]:
# создаём окружение
!conda create -n repeat_env -c bioconda -c conda-forge repeatmodeler repeatmasker -y

In [59]:
# создадим директории для промежуточных файлов, чтобы не мусорить

# RepeatModeler не имеет флага типа -outdir, он свои временные файлы и директории кладёт прямо туда,
# откуда запущен. Будем создавать ему отдельную директорию и заходить туда перед запуском

!mkdir -p repeats/modeler
!mkdir -p repeats/masker

# и для удобства путь к исходному геному
GENOME = "/home/miracle/bio/hw4/data/GCA_001949185.1_Rvar_4.0_genomic.fna"

In [60]:
# создаём БД для RepeatModeler

%cd repeats/modeler
!conda run -n repeat_env BuildDatabase -name yokozuma_db $GENOME

/home/miracle/bio/hw4/repeats/modeler
Building database yokozuma_db:
  Reading /home/miracle/bio/hw4/data/GCA_001949185.1_Rvar_4.0_genomic.fna...
Number of sequences (bp) added to database: 200 ( 55842812 bp )


In [61]:
# видим, что прочитано 200 последовательностей и 55842812 bp
# проверяем быстренько, что всё сходится:
!grep -v "^>" $GENOME | tr -d '\n' | wc -c

55842812


In [63]:
# и посмотрим, что с БД всё в порядке (BuildDatabase использует движок BLAST)

# если не установлен, устанавливаем blast:
# %conda install -c bioconda blast

!blastdbcmd -db yokozuma_db -info

Database: ./x1nV1JhIxr
	200 sequences; 55,842,812 total bases

Date: Apr 17, 2026  8:41 PM	Longest sequence: 9,333,084 bases

BLASTDB Version: 4

Volumes:
	/home/miracle/bio/hw4/repeats/modeler/yokozuma_db


In [64]:
# выходим обратно в корень проекта
%cd ../..

/home/miracle/bio/hw4


In [ ]:
%%bash
cd repeats/modeler

# запускаем поиск повторов de novo
# и засечём время ради интереса
# -pa 11, потому что у меня 12 потоков.
# в доках говорят, что лучше одно ядро под выполнение скрипта оставлять,
# а остальными делать поиск по BLAST
conda run -n repeat_env time RepeatModeler -database yokozuma_db -pa 11

Искало почти час на 11 потоках, нашло 391 семейство повторов.

In [ ]:
%%bash
cd repeats/masker
# каждая ячейка %%bash создаёт изолированную "среду оболочки", поэтому приходится заново задавать переменную:
GENOME="../../data/GCA_001949185.1_Rvar_4.0_genomic.fna"

# маскируем геном с помощью созданной библиотеки через RepeatMasker
# (-xsmall = soft-masking)
# -dir . = складывать всё в текущую папку
conda run -n repeat_env RepeatMasker -lib ../modeler/yokozuma_db-families.fa -xsmall -dir . \
    -pa 11 "$GENOME"

In [78]:
%%bash

# смотрим статистику маскирования
cd repeats/masker
cat *.tbl

file name: GCA_001949185.1_Rvar_4.0_genomic.fna
sequences:           200
total length:   55842812 bp  (55423672 bp excl N/X-runs)
GC level:         47.51 %
bases masked:    8333276 bp ( 14.92 %)
               number of      length   percentage
               elements*    occupied  of sequence
--------------------------------------------------
Retroelements          633       467214 bp    0.84 %
   SINEs:                0            0 bp    0.00 %
   Penelope:             0            0 bp    0.00 %
   LINEs:              533       372619 bp    0.67 %
    CRE/SLACS          303       176004 bp    0.32 %
     L2/CR1/Rex        155       142870 bp    0.26 %
     R1/LOA/Jockey       0            0 bp    0.00 %
     R2/R4/NeSL         35        13595 bp    0.02 %
     RTE/Bov-B          40        40150 bp    0.07 %
     L1/CIN4             0            0 bp    0.00 %
   LTR elements:       100        94595 bp    0.17 %
     BEL/Pao             0            0 bp    0.00 %
     Ty1/Copia    

Маскировка прошла успешно, замакскировали 8333276/55842812bp (14.92%)

### 2. Предсказание генов

Поскольку в базе данных AUGUSTUS нет модели, обученной на тихоходках, с помощью phyloT найдём ближайшего родственника из тех, для кого есть модели.

В [списке](https://bioinf.uni-greifswald.de/webaugustus/prediction/create) довольно много моделей, а бесплатная версия phyloT за раз принимает не более девяти видов (хотя на сайте написано, что не более десяти!). Поэтому для начала руками отберём наиболее вероятных кандидатов.

Наша тихоходка относится к кладе линяющих.  

Из списка моделей AUGUSTUS, к линяющим относятся членистоногие: Drosophila melanogaster, Apis mellifera, Bombus terrestris, Tribolium castaneum, Nasonia vitripennis, Camponotus floridanus, Heliconius melpomene, Acyrthosiphon pisum, Rhodnius prolixus, Aedes aegypti
И нематоды: Caenorhabditis elegans, Trichinella spiralis, Brugia malayi

Некоторых членистоногих отрезала, загружаю в [phyloT](https://phylot.biobyte.de/).  

Вот что в результате: 

![image.png](https://raw.githubusercontent.com/bimbomber/HW4_Genomic-Data-Analysis_2026/refs/heads/main/images/tree.png)

Верхняя ветвь - нематоды, круглые черви.  
Нижняя - членистоногие и наша тихоходка.  
Наглядно видно, что тихоходка отделилась от общего предка с членистоногими после того, как отошли нематоды. Поэтому берём насекомое. 

Выбираю Drosophila melanogaster, как наиболее изученный вид.

Далее, в том же [WebAUGUSTUS](https://bioinf.uni-greifswald.de/webaugustus/prediction/create), загружаем наш маскированный геном GCA_001949185.1_Rvar_4.0_genomic.masked.

В Select an organism выбираем дрозофилу.  
Start predicting.  
Ждём, когда завершится работа. Минут двадцать.

In [ ]:
%%bash
# создаём директорию для результатов предикции
mkdir -p augustus/drosophila_prediction
cd augustus/drosophila_prediction

# загружаем результаты
wget https://bioinf.uni-greifswald.de/webaugustus/prediction-results/pred_bftMJdW/predictions.tar.gz

# распаковываем
tar -xzvf *.tar.gz

In [ ]:
# посчитаем количество предсказанных белков
!grep -c "^>" augustus/drosophila_prediction/augustus/*.aa

11300


Завершили ab initio предсказание генов. Ближайшей моделью выбрали Дрозофилу.  
Нашли 11300 белков.

### 3. Обучение модели и реаннотация

Модель дрозофилы работает, но тихоходка всё-таки достаточно далека от неё эволюционно. Чтобы повысить точность предсказаний, натренируем AUGUSTUS на реальных кусочнах РНК нашей тихоходки (EST-последовательности), проведём повторную аннотацию и сравним результаты.

В тексте задания сказано, что EST следует скачивать из NCBI в формате GenBank.  
Я выбрала для тренировки веб-версию и, изучив документацию, поняла, что он на вход принимает сырой геном и сыре EST в формате fasta. Опционально можно предоставить "Training gene structure file" с точными координатами генов на хромосомах. Но мои файлы таких координат не имеют, это просто нарезанные куски РНК. 

И, если я правильно понимаю, в таком случае процесс происходит следующим образом: мы даём webAUGUSTUS сырой геном и сырой EST в fasta формате. Сервер сам запускает встроенный BLAST, накладывает эти EST на геном, вычисляет их координаты, сам создаёт этот самый Training gene structure file, и уже по нему обучается.

Может для локальной тренировки и нужны GenBank файлы, но документацию я не изучала. Так что не трогаю их.

Сначала попробовала такой путь получения нужных EST для обучения. Он оказался ошибочным. Оставляю ячейки кода для документации процесса, но выполнять их не нужно:

In [ ]:
# для начала нам надо собрать эти самые EST.
# возьмём их из базы NCBI EST.

# для удобства сделаем это через esearch, прямо здесь
!conda create -n entrez_env -c bioconda entrez-direct -y

!mkdir esearch

In [89]:
%%bash
cd esearch

# без такой инициализации conda были проблемы с broken pipe 
# а pipe нужен, поэтому инициализируем conda в этой сессии bash и всё работает
# (слава stack overflow)
eval "$(conda shell.bash hook)"
conda activate entrez_env

esearch -db nucest -query "Ramazzottius varieornatus[Organism]" | efetch -format fasta > rvar_est.fasta

curl: (56) OpenSSL SSL_read: OpenSSL/3.6.2: error:0A000126:SSL routines::unexpected eof while reading, errno 0
 ERROR:  curl command failed ( Sat Apr 18 15:57:34 MSK 2026 ) with: 56
-X POST https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi -d db=nucest&id=2255485236%2C2255485237%2C2255485238%2C2255485239%2C2255485240%2C2255485241%2C2255485242%2C2255485243%2C2255485244%2C2255485245%2C2255485246%2C2255485247%2C2255485248%2C2255485249%2C2255485250%2C2255485251%2C2255485252%2C2255485253%2C2255485254%2C2255485255%2C2255485256%2C2255485257%2C2255485258%2C2255485259%2C2255485260%2C2255485261%2C2255485262%2C2255485263%2C2255485264%2C2255485265%2C2255485266%2C2255485267%2C2255485268%2C2255485269%2C2255485270%2C2255485271%2C2255485272%2C2255485273%2C2255485274%2C2255485275%2C2255485276%2C2255485277%2C2255485278%2C2255485279%2C2255485280%2C2255485281%2C2255485282%2C2255485283%2C2255485284%2C2255485285&rettype=fasta&retmode=text&tool=edirect&edirect=25.3&edirect_os=Linux&email=miracle%40DE

Ячейка выше выполнялась 102 минуты и один раз вылетела с ошибкой curl

In [110]:
# посчитаем, что мы скачали
!grep -c "^>" esearch/rvar_est.fasta

140534


Но, несмотря на это, скачался очень большой fasta файл. В нём 140534 последовательностей.  
Здесь я заподозрила неладное и решила скачать через NCBI.  

Решила скачать дополнительно через браузер и проверить.

В тексте задания сказано "open the [R.varieornatus page](https://www.ncbi.nlm.nih.gov/Taxonomy/Browser/wwwtax.cgi?id=947166) in NCBI Taxonomy, select “Nucleotide EST”, download it...".  
Но сейчас такой ссылки там нет. Предполагаю, что была какая-то [реструктуризация NCBI](https://ncbiinsights.ncbi.nlm.nih.gov/2018/07/30/upcoming-changes-est-gss-databases/) и базу EST [ликвидировали как отдельную сущность](https://ncbiinsights.ncbi.nlm.nih.gov/2019/07/25/est-and-gss-databases-now-retired/).  

Если я правильно разобралась, сейчас валидный способ это сделать следующий:

Переходим на https://www.ncbi.nlm.nih.gov/nuccore  
Делаем поиск по БД "Nucleotide" с запросом `"Ramazzottius varieornatus"[Organism] AND est[Properties].`
Снизу Send to -> File -> Fasta -> Create File.  

Сразу замечаю, что файл загрузился куда меньшего размера.  
Переименовала его в "rvar_est_web.fasta" и поместила рядом с полученным с помощью esearch | efetch.

Проверила количество последовательностей:

In [111]:
!grep -c "^>" esearch/rvar_est_web.fasta

70819


Это заставило задуматься. После анализа (глазами и ctrl+f) обоих файлов сделала для себя вывод, что для нашей тихоходки есть два типа данных РНК: старые данные (по Сэнгеру, которые указаны в тексте задания) и новые данные (Illumina RNA-seq). 

Эти новые данные - TSA (Transcriptome Shotgun Asembly) - миллионы коротких ридов, собранных в огромный транскриптом. И этих TSA нет в полученных EST через веб-версию.

Я пыталась вводить в esearch | efetch фильтр `est[Properties]`, но тогда он не находил ничего. Поэтому я убрала этот фильтр, думая, что, добавив флаг `-db nucest`, ограничу выдачу только до нужных мне данных.

Веб-версия же вернула только Sanger-данные. Для тренировки буду использовать полученные через веб-версию данные.

Захожу на страничку [Training](http://bioinf.uni-greifswald.de/webaugustus/training/create) webAUGUSTUS. Указываю свою почту.  
Species name: Ramazzottius_varieornatus (пробел не разрешён)  
Genome file: загружаю маскированный геном  
cDNA file: загружаю компилированный est файл из web-версии ncbi  
Остальное - пусто.  
Ввожу капчу, соглашаюсь с условиями, Start Training.  


Спустя несколько часов, получила сообщение, что всё готово.

In [ ]:
%%bash 
# распакуем результаты из архивов

cd augustus/train_results/

for file in *.tar.gz; do tar -zxf "$file"; done

gunzip *.gb.gz

In [ ]:
%%bash

# убираю мусорные файлы чтоб не мозолили глаза
cd augustus/train_results/
rm -rf *.gz
find . -name "*:Zone.Identifier" -type f -delete

In [ ]:
# теперь можно посравнивать.
!echo количество генов, найденных с моделью мухи, до обучения модели:
!grep -c "^>" augustus/drosophila_prediction/augustus/*.aa

!echo количество генов, найденных только нашей новой моделью, без подсказок:
!grep -c "^>" augustus/train_results/ab_initio/*.aa

!echo количество генов, найденных моделью с подсказками EST:
!grep -c "^>" augustus/train_results/hints_pred/*.aa

!echo количество генов, найденных моделью с подсказками EST и попыткой предсказания UTR:
!grep -c "^>" augustus/train_results/hints_utr_pred/*.aa

количество генов, найденных с моделью мухи, до обучения модели:
11300
количество генов, найденных только нашей новой моделью, без подсказок:
11999
количество генов, найденных моделью с подсказками EST:
13777
количество генов, найденных моделью с подсказками EST и попыткой предсказания UTR:
10844


Я не уверена, насколько можно доверять качеству предсказания UTR (нетранслируемые области в конце генов), тем более при работе на коротких данных по Сэнгеру, поэтому для дальнейшей работы беру набор белков из hints_pred директории, 13777 найденных генов.

Делаем вывод, что теперь у нас количество генов на 21.9% больше, чем при поиске с моделью мухи.

Заглянем немного вперёд и загрузим precomputed результаты AUGUSTUS из задания 2. Structural annotation. 

[.aa](https://drive.google.com/file/d/1hCEywBlqNzTrIpQsZTVuZk1S9qKzqQAq/view?usp=sharing) и [.gff](https://drive.google.com/file/d/12ShwrgLkvJIYQV2p1UlXklmxSOOxyxj4/view) файлы. 

Сравним с нашими результатами:

In [112]:
# считаем количество белков в precomputed файле:
!grep -c "^>" augustus/precomp/*.aa

16435


Видим, что в precomputed результатах на 3 тысячи белков больше, чем в моих результатах из опциональной части.

Возможно, это из-за того, что я отказалась от локального пайплайна обучения и что-то пропустила.  
Возможно, из-за того, что отбросила Illumina данные для обучения.  
Возможно, я в чём-то не разобралась и предоставила недостаточно файлов для обучения.  
Возможно, замаскировала слишком много лишнего.


Ради эксперимента попробую переобучить на огромном количестве EST с TSA, полученных ранее через esearch | efetch.  
Не думаю, что получится что-то хорошее. Всё-таки там и куча коротких ридов, и полный геном, и перекрывающиеся последовательности. Может даже не досчитает до дедлайна. Или переобучится. 

Просто посмотрю, что получится, если получится.

In [113]:
!grep -c "^>" esearch/rvar_est.fasta

140534


Первая проблема была с тем, что webAUGUSTUS позволяет загрузить лишь файл до 100МБ. Наш - 190МБ. Нужно было найти хостинг, предсотавляющий прямые ссылки на загруженные файлы. Перебрав десятки, сработал только [MediaFire](https://www.mediafire.com/).

Обучение шло 20 часов. В результате получились такие цифры:

In [73]:
!echo количество генов, найденных только нашей новой моделью, без подсказок:
!grep -c "^>" augustus/train_results_efetch/ab_initio/*.aa

!echo количество генов, найденных моделью с подсказками EST:
!grep -c "^>" augustus/train_results_efetch/hints_pred/*.aa

!echo количество генов, найденных моделью с подсказками EST и попыткой предсказания UTR:
!grep -c "^>" augustus/train_results_efetch/hints_utr_pred/*.aa

количество генов, найденных только нашей новой моделью, без подсказок:
12232
количество генов, найденных моделью с подсказками EST:
17011
количество генов, найденных моделью с подсказками EST и попыткой предсказания UTR:
12943


Для дальнейшего анализа всё равно буду использовать модель, обученную на чистых данных Сэнгера, но в конце попробую прогнать через получившийся пайплайн и эту модель, чтобы сравнить результаты.

## Неопциональная часть

### 2. Структурная аннотация

In [ ]:
# Count the number of obtained proteins

# How can we narrow it down to a list of potential candidates 
# that we can then verify by experimentation?

Загружаем precomputed результаты AUGUSTUS из задания.  

[.aa](https://drive.google.com/file/d/1hCEywBlqNzTrIpQsZTVuZk1S9qKzqQAq/view?usp=sharing) и [.gff](https://drive.google.com/file/d/12ShwrgLkvJIYQV2p1UlXklmxSOOxyxj4/view) файлы. 

Так как загружены файлы и .gff и .aa, пропускаем этап извлечения белков из gff. 

In [114]:
# считаем количество белков в precomputed файле:
!grep -c "^>" augustus/precomp/*.aa

16435


Дальнейший анализ всё-таки будем проводить к precomputed данными, ведь белков здесь выше и качество наверняка лучше. Как бы ни было жалко потраченного времени и компьюта на опциональные шаги выше.

Чтобы сузить это количество белков до кандидатов, связанных с восстановлением ДНК, мы можем взять данные масс-спектрометрии белков, выделенных из хроматиновой фракции тихоходки (реальные короткие пептиды), и выровнять их на нашу базу из 16435 предсказанных белков. Те белки, которые будут содержать экспериментально подтвержденные пептиды, и составят финальный список кандидатов для дальнейшей проверки.

### 3. Физическая локализация

Итак, у нас есть гипотеза: радиационная устойчивость может обеспечиваться определёнными белками, связанными с ДНК. Коллеги выделили хроматиновую фракцию, провели масс-спектрометрию и получили список коротких пептидов, связанных с ДНК.

Теперь наша задача - найти, каким белкам из нашего генома принадлежат эти пептиды.  

Будем использовать diamond (в 1000 раз быстрее BLASTа) для поиска выравниваний пептидов на нашей базе предсказанных белков.  

После этого извлечём полные последовательности этих кандидатных белков с помощью python

In [ ]:
# создаём окружение
!conda create -n diamond_env -c bioconda -c conda-forge diamond -y

In [ ]:
# создаём директорию для этого шага
!mkdir -p diamond

И в эту же директорию загружаем [peptides.faa](https://disk.yandex.ru/d/xJqQMGX77Xueqg)

In [ ]:
%%bash
cd diamond

# создаём бд из всех белков тихоходки
conda run -n diamond_env diamond makedb --in ../augustus/precomp/*.aa --db rvar_proteins_db

# выполняем поиск пептидов 
conda run -n diamond_env diamond blastp -d rvar_proteins_db.dmnd -q peptides.fa \
    -f 6 \
    -o diamond_results.tsv --very-sensitive

А в следующей ячейке проверю тот же ран diamond, но не на precomputed данных, а на результатах аннотации AUGUSTUS от собственной обученной модели.

In [ ]:
%%bash
mkdir -p diamond_my
cd diamond_my

# создаём бд из всех белков тихоходки
conda run -n diamond_env diamond makedb --in ../augustus/train_results/hints_pred/*.aa --db rvar_my_proteins_db

# выполняем поиск пептидов 
conda run -n diamond_env diamond blastp -d rvar_my_proteins_db.dmnd -q peptides.fa \
    -f 6 \
    -o diamond_results_my.tsv --very-sensitive

Сравним результаты. Помним, что мы потеряли чуть меньше 3000 белков при нашей аннотации, в сравнении с precomputed аннотацией.

In [27]:
import pandas as pd
from Bio import SeqIO
from IPython.display import display

columns = [
    'qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 
    'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore'
]

df_precomp = pd.read_csv('diamond/diamond_results.tsv', sep='\t', names=columns)
df_my = pd.read_csv('diamond_my/diamond_results_my.tsv', sep='\t', names=columns)

print("Результаты diamond (precomputed AUGUSTUS)")
display(df_precomp)
print("\nРезультаты diamond (моя модель AUGUSTUS)")
display(df_my)

# далее извлекаем последовательности кандидатов
# берём id целевых белков из precomputed базы (только 100% совпадение)
filtered_hits = df_precomp[(df_precomp['evalue'] < 1e-3) & (df_precomp['pident'] == 100.0)]
candidate_ids = set(filtered_hits['sseqid'].tolist())

print(f"\nНайдено уникальных белков-кандидатов: {len(candidate_ids)}")
print(f"IDs: {candidate_ids}")

candidates_records = []
protein_fasta_file = "augustus/precomp/augustus.whole.aa" 

for record in SeqIO.parse(protein_fasta_file, "fasta"):
    if record.id in candidate_ids:
        candidates_records.append(record)

output_file = "diamond/candidate_proteins.faa"
SeqIO.write(candidates_records, output_file, "fasta")

my_candidates_records = []
my_protein_fasta_file = "augustus/train_results/hints_pred/augustus.aa" 

for record in SeqIO.parse(my_protein_fasta_file, "fasta"):
    if record.id in candidate_ids:
        my_candidates_records.append(record)

output_file = "diamond_my/candidate_proteins_my.faa"
SeqIO.write(candidates_records, output_file, "fasta")

Результаты diamond (precomputed AUGUSTUS)


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,8,g4106.t1,100,18,0,0,1,18,222,239,5.550000e-07,40.8
1,20,g12510.t1,100,18,0,0,1,18,425,442,1.430000e-06,39.7
2,21,g12510.t1,100,22,0,0,1,22,443,464,2.460000e-09,47.8
3,29,g4106.t1,100,18,0,0,1,18,222,239,5.550000e-07,40.8



Результаты diamond (моя модель AUGUSTUS)


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,8,g3280.t1,100,18,0,0,1,18,234,251,5.180000e-07,40.8
1,20,g10549.t1,100,18,0,0,1,18,106,123,1.240000e-06,39.7
2,21,g10549.t1,100,22,0,0,1,22,124,145,1.610000e-09,47.8
3,29,g3280.t1,100,18,0,0,1,18,234,251,5.180000e-07,40.8



Найдено уникальных белков-кандидатов: 2
IDs: {'g12510.t1', 'g4106.t1'}


2

Во-первых, видим, что потерянные белки, не аннотированные моей моделью, по сравнению с precomputed, не повлияли на результат - они нам не полезны. Результаты идентичны, за исключением ids (с моей моделью пропущены определённые белки, поэтому и числа в id меньше). Из-за этого считаю, что этап с тренировкой модели всё-таки был успешен и был не зря.

Во-вторых, извлекаем в candidate_proteins.faa полные аминокислотные последовательности уникальных найденных кандидатных белков (g4106.t1 и g12510.t1 по аннотации из precomputed модели) для дальнейшего функционального аннотирования.  
И, на всякий случай, аннотированные из собственной AUGUSTUS модели, тоже. В директории diamond/ и diamond_my/, соответственно.

Но, забегая вперёд, следующий шаг в тексте задания начинается с фразы "There are still too many proteins...".  
Меня это смутило, так как diamond выдал всего 2 белка. Поэтому напишу python скрипт, чтобы перепроверить. 

Ищем просто 100%ные вхождения подстроки в строку.

In [25]:
# для precomputed результата:
from Bio import SeqIO

def search_proteins(filepath, output):
    proteins_file = filepath
    peptides_file = "diamond/peptides.fa"

    proteins = {rec.id: str(rec.seq) for rec in SeqIO.parse(proteins_file, "fasta")}
    peptides = {rec.id: str(rec.seq) for rec in SeqIO.parse(peptides_file, "fasta")}

    print(f"белков: {len(proteins)}")
    print(f"пептидов: {len(peptides)}")

    exact_matches = []
    unique_candidates = set()

    for pep_id, pep_seq in peptides.items():
        # убираем возможные символы переноса и делаем uppercase на всякий случай
        clean_pep = pep_seq.strip().upper() 
        
        for prot_id, prot_seq in proteins.items():
            if clean_pep in prot_seq.upper():
                exact_matches.append({'peptide_id': pep_id, 'protein_id': prot_id})
                unique_candidates.add(prot_id)

    print(f"\nточных совпадений пептид-белок: {len(exact_matches)}")
    print(f"уникальных белков-кандидатов: {len(unique_candidates)}\n")

    for i, cand in enumerate(unique_candidates):
        print(cand)
        
    candidate_records = []

    for record in SeqIO.parse(proteins_file, "fasta"):
        if record.id in unique_candidates:
            candidate_records.append(record)

    # да, немного нелогично сохранять в директорию diamond, но мне так удобнее. Весь этот шаг в ней работаем. 
    SeqIO.write(candidate_records, output, "fasta")


In [26]:
# для precomputed результата:
proteins_file = "augustus/precomp/augustus.whole.aa"
search_proteins(proteins_file, "diamond/candidate_proteins_python.faa")

белков: 16435
пептидов: 43

точных совпадений пептид-белок: 62
уникальных белков-кандидатов: 19

g5237.t1
g702.t1
g5503.t1
g5510.t1
g4106.t1
g15484.t1
g12562.t1
g13530.t1
g12510.t1
g10513.t1
g3679.t1
g5641.t1
g14472.t1
g3428.t1
g5502.t1
g1285.t1
g5616.t1
g10514.t1
g15153.t1


In [27]:
# и для аннотации нашей моделью:
proteins_file = "augustus/train_results/hints_pred/augustus.aa" 
search_proteins(proteins_file, "diamond_my/candidate_proteins_python_my.faa")

белков: 13777
пептидов: 43

точных совпадений пептид-белок: 62
уникальных белков-кандидатов: 20

g10549.t1
g12173.t1
g10550.t1
g4341.t1
g8802.t1
g4558.t1
g3280.t1
g1132.t1
g4551.t1
g4552.t1
g11427.t1
g2933.t1
g12739.t1
g10595.t1
g4636.t1
g2752.t1
g8803.t1
g4653.t1
g13008.t1
g581.t1


Собственно, опасения подтвердились. Diamond нашёл очень мало белков. Возможно, это связано с тем, что он посчитал некоторые совпадения случайными на основе своей математики, так как пептиды очень короткие. Для дальнейшего анализа буду использовать белки, найденные собственным python-скриптом, так как он точно честно нашёл все вхождения, ничего не фильтруя.

Так же заметила, что моя модель (13к белков) нашла 20 кандидатов, а precomputed (16к белков) - 19. Скорее всего, это связано с фрагментацией. Моя модель могла ошибиться с границами экзонов или интронов и какой-нибудь важный белок разорвать на два отдельных коротких белка, из-за чего пептиды могли попасть на две половинки этого белка. Сохраню результаты, основнанные на обоих аннотациях, и для дальнейшего анализа, чтобы наверняка, буду вести две ветки анализа: и precomputed результат, и данные собственной модели.

И запомню белки из Diamond, как возможное доказательство.

### 4. Предсказание локализаций

В итоге получилось 19(precomputed)/20(моя модель) белков. Нужно попытаться предсказать, где эти белки локализованы в клетке.  
Поскольку мы ищем белок, который защищает ДНК или учавствует в восстановлении, мы можем ожидать, что он будет находиться в ядре.

[__WoLF PSORT__](https://wolfpsort.hgc.jp/).

Используем для предсказания финального места назначения белка (ядро, цитоплазма, мембрана и т.д.).

Считаем это позитивным фильтром: нужен маркер nucl (ядро) с высоким баллом.

Указываем Animal, вставляем fasta файл из python-скрипта, сохраняем HTML результат в localization/wolfsport_pretrain.html



In [ ]:
import re
import pandas as pd
from IPython.display import display

def parse_wolf(filepath):
    with open(filepath) as f:
        wolf_html = f.read()

    # ID -> тег <a> -> список баллов
    pattern = re.compile(r"(g\d+\.t1)\s+<a[^>]+>details</a>\s+([^<]+)")

    parsed_data = []
    for match in pattern.finditer(wolf_html):
        prot_id = match.group(1)
        scores = match.group(2).strip()
        top_loc = scores.split(':')[0].strip()
        
        parsed_data.append({
            'ID': prot_id,
            'Top Localization': top_loc,
            'All Scores': scores
        })

    df_wolf = pd.DataFrame(parsed_data)

    others = sorted([x for x in df_wolf['Top Localization'].unique() if x not in ['nucl']])
    df_wolf['Top Localization'] = pd.Categorical(df_wolf['Top Localization'], categories=['nucl'] + others, ordered=True)
    df_wolf = df_wolf[['ID', 'All Scores', 'Top Localization']]
    
    return df_wolf.sort_values('Top Localization')

In [105]:
# посмотрим на результат (pretrain proteins)
pretrain_wolf = parse_wolf('localization/wolfpsort_pretrain.html')
display(pretrain_wolf)

,ID,All Scores,Top Localization
18,g15484.t1,"nucl: 17.5, cyto_nucl: 15.3333, cyto: 12, cyto...",nucl
16,g14472.t1,"nucl: 28, plas: 2, cyto: 1, cysk: 1",nucl
12,g10514.t1,"nucl: 19, cyto_nucl: 15, cyto: 9, extr: 3, mit...",nucl
11,g10513.t1,"nucl: 20, cyto_nucl: 14.5, cyto: 7, extr: 3, E...",nucl
4,g4106.t1,"E.R.: 14.5, E.R._golg: 9.5, extr: 7, golg: 3.5...",E.R.
15,g13530.t1,"extr: 13, nucl: 6.5, lyso: 5, cyto_nucl: 4.5, ...",extr
14,g12562.t1,"extr: 30, lyso: 2",extr
10,g5641.t1,"extr: 31, lyso: 1",extr
17,g15153.t1,extr: 32,extr
0,g702.t1,"extr: 29, plas: 2, lyso: 1",extr


In [35]:
# посмотрим на результат (self-trained proteins)
my_wolf = parse_wolf('localization/wolfpsort_my.html')
display(my_wolf)

,ID,All Scores,Top Localization
19,g13008.t1,"nucl: 17, cyto: 12.5, cyto_mito: 7, plas: 1, g...",nucl
17,g12173.t1,"nucl: 28, plas: 2, cyto: 1, cysk: 1",nucl
15,g10595.t1,"nucl: 23, cyto: 7, plas: 2",nucl
12,g8803.t1,"nucl: 26, cyto_nucl: 16.5, cyto: 5, cysk_plas: 1",nucl
16,g11427.t1,"extr: 14, lyso: 6, nucl: 3.5, plas: 3, E.R.: 3...",extr
18,g12739.t1,extr: 32,extr
7,g4552.t1,"extr: 17, nucl: 8, cyto: 6, lyso: 1",extr
0,g581.t1,"extr: 29, plas: 2, lyso: 1",extr
3,g2933.t1,"extr: 26, mito: 2, lyso: 2, plas: 1, E.R.: 1",extr
1,g1132.t1,"extr: 25, plas: 5, mito: 1, lyso: 1",extr


Здесь сразу обращаем внимание, что белки g4106.t1/g12510.t1 (в моей модели g3280.t1/g10549.t1), которые выделил Diamond, на самом деле идут в плазматическую мембрану и эндоплазматический ретикулум.

[__TargetP Server__](https://services.healthtech.dtu.dk/services/TargetP-2.0/).

Используем для определения вероятности того, что белок является секретируемым или митохондриальным.

Считаем это негативным фильтром. Если белок уходит в митохондрии или секретируется наружу, он не сможет защищать ДНК.

Указываем Non-Plant, Submit. Сохраняем результаты в localization/targetp_pretrain.txt (csv-formatted)

In [30]:
import pandas as pd
import io
from IPython.display import display

def parse_targetp(filepath):
    with open(filepath) as f:
        targetp_txt = f.read()

    # читаем как таблицу, игнорируя строки с комментариями (#), кроме заголовков
    df_targetp = pd.read_csv(io.StringIO(targetp_txt), sep='\t', comment='#', names=['ID', 'Prediction', 'OTHER', 'SP', 'mTP', 'CS Position'])

    df_targetp.fillna('', inplace=True)
    
    return df_targetp.sort_values('OTHER', ascending=False)

In [106]:
# посмотрим на результаты (для pretrain proteins)
pretrain_targetp = parse_targetp('localization/targetp_pretrain.txt')
display(pretrain_targetp)

,ID,Prediction,OTHER,SP,mTP,CS Position
16,g14472.t1,OTHER,0.999999,0.000001,0.000000,
11,g10513.t1,OTHER,0.999999,0.000001,0.000000,
18,g15484.t1,OTHER,0.999980,0.000010,0.000010,
2,g3428.t1,OTHER,0.999903,0.000033,0.000064,
13,g12510.t1,OTHER,0.999738,0.000099,0.000163,
5,g5237.t1,OTHER,0.999545,0.000345,0.000111,
12,g10514.t1,OTHER,0.999543,0.000349,0.000107,
8,g5510.t1,OTHER,0.999108,0.000016,0.000876,
4,g4106.t1,OTHER,0.729658,0.266917,0.003425,
15,g13530.t1,SP,0.116007,0.883840,0.000153,CS pos: 19-20. TIP-FT. Pr: 0.3552


In [33]:
# посмотрим на результаты (для self-trained proteins)
my_targetp = parse_targetp('localization/targetp_my.txt')
display(my_targetp)

,ID,Prediction,OTHER,SP,mTP,CS Position
17,g12173.t1,OTHER,0.999999,0.000001,0.000000,
19,g13008.t1,OTHER,0.999980,0.000010,0.000010,
15,g10595.t1,OTHER,0.999959,0.000007,0.000033,
2,g2752.t1,OTHER,0.999903,0.000033,0.000064,
11,g8802.t1,OTHER,0.999836,0.000045,0.000119,
10,g4653.t1,OTHER,0.999488,0.000015,0.000497,
8,g4558.t1,OTHER,0.999108,0.000016,0.000876,
9,g4636.t1,OTHER,0.997258,0.002727,0.000015,
14,g10550.t1,OTHER,0.994087,0.000064,0.005849,
13,g10549.t1,OTHER,0.992576,0.001304,0.006121,


### 5. Поиск в BLAST

Теперь сделаем поиск в [BLAST](https://blast.ncbi.nlm.nih.gov/Blast.cgi?PROGRAM=blastp&PAGE_TYPE=BlastSearch&LINK_LOC=blasthome). Вставляем нашу последовательность (candidate_proteins.faa), выбираем БД UniProtKB/Swiss-Prot и ищем.

В первую очередь нас интересуют последовательности, которые нигде не встречаются. Загружаем результат в формате single-file JSON в директорию blast.

In [1]:
# распарсим json, чтобы сохранить Accession Number, E-value, % Ident, % Query coverage, and annotation.
import json
import pandas as pd
from IPython.display import display

def parse_blast_json(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)

    results = []

    searches = data.get('BlastOutput2', [])

    for search_wrapper in searches:
        # Извлекаем данные запроса
        search = search_wrapper.get('report', {}).get('results', {}).get('search', {})
        query_title = search.get('query_title', search.get('query_id', 'Unknown'))
        query_len = search.get('query_len', 1)

        hits = search.get('hits', [])

        # лучшие кандидаты
        if not hits:
            results.append({
                'ID': query_title, # id белка
                'Accession': None,
                'Annotation': None,
                'E-value': None,
                '% Ident': None,
                '% Coverage': None,
                'Species': None
            })
            continue

        # берём только best hit
        best_hit = hits[0]
        description = best_hit.get('description', [{}])[0]
        accession = description.get('accession', 'Unknown')
        annotation = description.get('title', 'Unknown')
        species = description.get('sciname', "Unknown")

        # берём лучшее выравнивание (HSP) из этого хита
        best_hsp = best_hit.get('hsps', [{}])[0]
        evalue = best_hsp.get('evalue')
        identity = best_hsp.get('identity', 0)
        align_len = best_hsp.get('align_len', 1)
        query_from = best_hsp.get('query_from', 0)
        query_to = best_hsp.get('query_to', 0)

        # Рассчитываем проценты
        pct_ident = round((identity / align_len) * 100, 2) if align_len else 0
        pct_cov = round(((query_to - query_from + 1) / query_len) * 100, 2) if query_len else 0

        results.append({
            'ID': query_title.split()[0],
            'Accession': accession,
            'Annotation': annotation,
            'E-value': evalue,
            '% Ident': pct_ident,
            '% Coverage': pct_cov,
            'Species': species
        })
    
    return results

In [104]:
# для pretrain:
pretrain_blast_df = pd.DataFrame(parse_blast_json('blast/pretrain-Alignment.json'))
display(pretrain_blast_df.sort_values('% Coverage', ascending=False))

,% Coverage,% Ident,Accession,Annotation,E-value,ID,Species
16,100.00,100.00,P0DOW4,RecName: Full=Damage suppressor protein [Ramaz...,0.000000e+00,g14472.t1,Ramazzottius varieornatus
2,91.81,56.60,Q09510,RecName: Full=Myosin regulatory light chain; A...,8.778600e-65,g3428.t1,Caenorhabditis elegans
18,78.18,45.03,Q155U0,RecName: Full=Vacuolar protein sorting-associa...,0.000000e+00,g15484.t1,Danio rerio
3,72.98,29.72,Q19269,RecName: Full=Zinc metalloproteinase nas-14; A...,7.443390e-22,g3679.t1,Caenorhabditis elegans
17,46.89,39.76,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,1.891390e-14,g15153.t1,Ethmostigmus rubripes
1,44.04,37.21,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,1.790140e-12,g1285.t1,Ethmostigmus rubripes
10,43.23,39.29,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,5.120040e-13,g5641.t1,Ethmostigmus rubripes
14,41.50,39.76,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,7.166190e-13,g12562.t1,Ethmostigmus rubripes
9,40.59,40.96,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,2.251740e-14,g5616.t1,Ethmostigmus rubripes
0,39.07,40.48,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,1.495040e-11,g702.t1,Ethmostigmus rubripes


In [5]:
# для нашей обученной модели:
my_blast_df = pd.DataFrame(parse_blast_json('blast/my-Alignment.json'))
display(my_blast_df.sort_values('% Coverage', ascending=False))

,% Coverage,% Ident,Accession,Annotation,E-value,ID,Species
17,100.00,100.00,P0DOW4,RecName: Full=Damage suppressor protein [Ramaz...,0.000000e+00,g12173.t1,Ramazzottius varieornatus
2,91.81,56.60,Q09510,RecName: Full=Myosin regulatory light chain; A...,8.778600e-65,g2752.t1,Caenorhabditis elegans
19,77.62,46.12,Q155U0,RecName: Full=Vacuolar protein sorting-associa...,0.000000e+00,g13008.t1,Danio rerio
3,74.04,27.70,O62243,RecName: Full=Zinc metalloproteinase nas-1; Al...,3.810480e-22,g2933.t1,Caenorhabditis elegans
10,53.87,28.14,Q96DM1,RecName: Full=PiggyBac transposable element-de...,1.756680e-29,g4653.t1,Homo sapiens
15,53.56,38.60,Q6DRC4,RecName: Full=Eukaryotic translation initiatio...,6.124900e-52,g10595.t1,Danio rerio
18,46.89,39.76,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,1.891390e-14,g12739.t1,Ethmostigmus rubripes
1,44.04,37.21,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,1.790140e-12,g1132.t1,Ethmostigmus rubripes
0,39.07,40.48,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,1.419470e-11,g581.t1,Ethmostigmus rubripes
7,38.86,39.76,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,5.791500e-14,g4552.t1,Ethmostigmus rubripes


### 6. Pfam предсказание

Даже если белок не похож ни на что известное, можем найти известные домены.

Идём в [HMMER](https://www.ebi.ac.uk/Tools/hmmer/search/hmmscan), hmmscan, загружаем последовательность, database Pfam, submit.

В интерфейсе нет опции загрузить сразу все результаты, поэтому загружаем .tsv результаты только для неизвестных BLASTу белков, которые TargetP определил как OTHER, а WoLF как nucl.

In [77]:
# pretrain candidates:
blast_set = set(pretrain_blast_df[pretrain_blast_df['E-value'].isna() | (pretrain_blast_df['E-value'] > 0.01)]['ID'])
targetp_set = set(pretrain_targetp[pretrain_targetp['Prediction'] == 'OTHER']['ID'])
wolf_set = set(pretrain_wolf[pretrain_wolf['Top Localization'] == 'nucl']['ID'])

print(blast_set & targetp_set & wolf_set)

{'g10513.t1', 'g10514.t1'}


In [76]:
# my_model candidates:
blast_set = set(my_blast_df[my_blast_df['E-value'].isna() | (my_blast_df['E-value'] > 0.01)]['ID'])
targetp_set = set(my_targetp[my_targetp['Prediction'] == 'OTHER']['ID'])
wolf_set = set(my_wolf[my_wolf['Top Localization'] == 'nucl']['ID'])

print(blast_set & targetp_set & wolf_set)

{'g8803.t1'}


Но для этих белков нет совпадений. Ни один из них не имеет классического структурного домена. Скачивать просто нечего.

А так как мы знаем, что все эти белки (1 из моей модели, 2 из предобученной) сидят на ДНК (пептиды сошлись на 100%), точно локализуются в ядре (WoLF: nucl), точно не уходят в другие компоненты (TargetP: OTHER), не имеют родственников (BLAST: No hits)...

Делаем вывод, что перед нами именно орфанный ген. Белок, который тихоходка изобрела с нуля в ходе эволюции.


### Предытог по двум веткам анализа

Считаю, что самое время разобраться, чем отличается моя модель от предобученной

In [ ]:
# сначала достанем аминокислотные последовательности для каждого из белков
from Bio import SeqIO

def get_sequence_by_id(fasta_file, target_id):
    with open(fasta_file, "r") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            if record.id == target_id:
                return str(record.seq)

my_sequence = get_sequence_by_id("diamond_my/candidate_proteins_python_my.faa", "g8803.t1")
pretrain_sequences = [get_sequence_by_id("diamond/candidate_proteins_python.faa", "g10513.t1"), \
    get_sequence_by_id("diamond/candidate_proteins_python.faa", "g10514.t1")]


In [82]:
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

# сравниваем наш белок с первым белком предобученной модели
print("my (g8803.t1) vs pretrain 1 (g10513.t1)")
# глобальное выравнивание: совпадение=1, несовпадение=0
alignments_1 = pairwise2.align.globalxx(my_sequence, pretrain_sequences[0])
print(format_alignment(*alignments_1[0]))

# и со вторым
print("my (g8803.t1) vs pretrain 2 (g10514.t1)")
alignments_2 = pairwise2.align.globalxx(my_sequence, pretrain_sequences[1])
print(format_alignment(*alignments_2[0]))

my (g8803.t1) vs pretrain 1 (g10513.t1)
MFINDV-T--SANFQFGPEVKRRKGIKGQQNSLSSSLR-----SFL----VLLSAFRHSSC----SRGYLVLFQVRQRPRSRFAMSYNRTEYR--SDSDRHDDDKQGGWRSWFGL---GGNKD----DNN-DR-DRGY----S--Y-T---NTTT--YRGDD-S------NR------YGF-SGDRMS-GP--SG------Y--SGAI--S-G--GY--S-G-SRG---GY--GYTTS----------------------D------D------YN--RGNT---NYSR-----SNDNSN----YNRDN-SSRG-GDRDVYYRQETRTYPTTG--YTG-SSNYTSGNYG--S--YGTP--S--YGTP--SYGT--Q-S--FGPQ-S--FGP--S---YGNTDRD------YTV---------SRTTT---NYNTDRD--Y-------------RPTGTT-GTTG-YGFSGDR-DTSFRNTSFNTTGDRDTFNR-GSYSG-GYAT-SGNQG-G-------FSTTSGMGHPGNYSSSYTS-P--SGTYGQSSN-Y---SYN-RNY-
|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

1 выравнивание - очевидно, это разные гены. 

2 выравнивание - уже интереснее. Это один и тот же белок, только в представлении моей модели он имеет 68 дополнительных аминокислот на N-конце. Возможно, моя модель предсказала истинный старт-кодон и собрала полноразмерную, нефрагментированную версию орфанного белка g8803.t1.

### 7. Сборка доказательств

In [ ]:
# a) best blast hit (annotation and e-value)
# b) predicted Pfam domains
# c) probable localization(s) according to WoLF PSORT
# d) localization according to TargetP

# What entries can you recommend for experimental verification as novel tardigrade-specific
# DNA-binding proteins? For all database hits pay special attention to e-value, and describe
# reliability of your findings.

|ID Белка                     |Лучший хит BLAST (Аннотация)                               |E-value|Домены Pfam  |Локализация WoLF PSORT|Локализация TargetP|
|-----------------------------|-----------------------------------------------------------|-------|-------------|----------------------|-------------------|
|g12173.t1 (My)<br />g14472.t1 (Pre)|Damage suppressor protein (Dsup)[Ramazzottius varieornatus]|0.0    |Не обнаружены|nucl (28)             |OTHER              |
|g8803.t1 (My)                |Нет совпадений (Орфанный ген)                              |N/A    |Не обнаружены|nucl (26)             |OTHER              |
|g10514.t1 (Pre)              |Нет совпадений (Орфанный ген)                              |N/A    |Не обнаружены|nucl (19)             |OTHER              |
|g10513.t1 (Pre)              |Нет совпадений (Орфанный ген)                              |N/A    |Не обнаружены|nucl (20)             |OTHER              |


Остальные белки локализуются преимущественно в экзосомах (extr), плазмалемме (plas) или митохондриях (mito), либо имеют уверенные гомологи, не связанные с репарацией ДНК (например, Myosin, E-value ~10^-65 ). Их мы исключаем из фокуса нашего внимания.

Построенный пайплайн успешно выявил белок g12173.t1/g14472.t1, который по результатам BLAST (E-value 0.0) однозначно идентифицируется как Damage suppressor protein (Dsup). Этот белок локализуется в ядре (WoLF score: 28) и специфичен для тихоходок. Это подтверждает, что использованные хроматиновые фракции и масс-спектрометрия действительно обогащены нужными нам защитными белками.

Наибольший интерес для дальнейшей экспериментальной проверки представляет белок g8803.t1 (и его усечённая версия g10514.t1 из precomputed-сборки). Физически связан с ДНК (подтверждено пептидами), имеет ярко выраженную ядерную локализацию (WoLF score: 26) и не предсказывается как секреторный или митохондриальный (TargetP: OTHER), полное отсутствие гомологов в базе Swiss-Prot, отсутствие известных функциональных доменов Pfam. Все основания полагать, что это истинный орфанный ген, возникшией de novo в тихоходках.

Кроме этого, сравнение предсказаний показало, что модель, обученная на собственных RNA-seq данных (cDNA), справилась лучше базовой. Белок g8803.t1 содержит дополнительный N-концевой домен из 68 аминокислот по сравнению с g10514.t1, что с высокой долей вероятности указывает на более точное предсказание старт-кодона и сборку полноразмерного функционального белка.

### Дополнительный эксперимент

In [123]:
# для Discussion секции

Тот же пайплайн. 

Модель, обученная на Illumina RNA-seq (TSA) данных.  
17011 найденных генов (против 16435 precomputed).  

Подводные камни: TSA-данные содержат множество изоформ (вариантов альтернативного сплайсинга) одного и того же гена. Скармливая их AUGUSTUS, алгоритм может запутаться и попытаться предсказать изоформы как отдельные, фрагментированные гены, идущие подряд. Предполагаю, что это причина, почему вышло чуть больше генов (17к), чем в эталонном наборе (16.4к).

Быстрый взгляд на размеры моделей:  

муха - 11300 генов  
обученная на Sanger - 13777 генов  
обученная на Sanger+TSA - 17011 генов  


In [101]:
# пептиды
from Bio import SeqIO

def search_proteins(filepath, output):
    proteins_file = filepath
    peptides_file = "diamond/peptides.fa"

    proteins = {rec.id: str(rec.seq) for rec in SeqIO.parse(proteins_file, "fasta")}
    peptides = {rec.id: str(rec.seq) for rec in SeqIO.parse(peptides_file, "fasta")}

    print(f"белков: {len(proteins)}")
    print(f"пептидов: {len(peptides)}")

    exact_matches = []
    unique_candidates = set()

    for pep_id, pep_seq in peptides.items():
        # убираем возможные символы переноса и делаем uppercase на всякий случай
        clean_pep = pep_seq.strip().upper() 
        
        for prot_id, prot_seq in proteins.items():
            if clean_pep in prot_seq.upper():
                exact_matches.append({'peptide_id': pep_id, 'protein_id': prot_id})
                unique_candidates.add(prot_id)

    print(f"\nточных совпадений пептид-белок: {len(exact_matches)}")
    print(f"уникальных белков-кандидатов: {len(unique_candidates)}\n")

    for i, cand in enumerate(unique_candidates):
        print(cand)
        
    candidate_records = []

    for record in SeqIO.parse(proteins_file, "fasta"):
        if record.id in unique_candidates:
            candidate_records.append(record)

    # да, немного нелогично сохранять в директорию diamond, но мне так удобнее. Весь этот шаг в ней работаем. 
    SeqIO.write(candidate_records, output, "fasta")
    
search_proteins("augustus/train_results_efetch/hints_pred/augustus.aa", "candidates_17k.fasta")


белков: 17011
пептидов: 43

точных совпадений пептид-белок: 62
уникальных белков-кандидатов: 20

g5438.t1
g739.t1
g5712.t1
g14932.t1
g15627.t1
g10874.t1
g13005.t1
g5703.t1
g10875.t1
g3725.t1
g5705.t1
g1408.t1
g5811.t1
g5835.t1
g15984.t1
g4156.t1
g14016.t1
g12947.t1
g12948.t1
g3481.t1


In [107]:
new_wolf = parse_wolf('localization/wolfpsort_17k.html')
display(new_wolf)

new_targetp = parse_targetp('localization/targetp_17k.txt')
display(new_targetp)

new_blast_df = pd.DataFrame(parse_blast_json('blast/17k.json'))
display(new_blast_df.sort_values('% Coverage', ascending=False))

,ID,All Scores,Top Localization
17,g14932.t1,"nucl: 28, plas: 2, cyto: 1, cysk: 1",nucl
15,g13005.t1,"nucl: 23, cyto: 7, plas: 2",nucl
12,g10875.t1,"nucl: 19.5, cyto_nucl: 13, extr: 6, cyto: 5.5,...",nucl
11,g10874.t1,"nucl: 10.5, cyto_nucl: 9.5, mito: 9, cyto: 5.5...",nucl
0,g739.t1,"extr: 29, plas: 2, lyso: 1",extr
16,g14016.t1,"extr: 14, lyso: 8, nucl: 4.5, cyto_nucl: 3.5, ...",extr
10,g5835.t1,"extr: 26, mito: 5, plas: 1",extr
18,g15627.t1,extr: 32,extr
9,g5811.t1,"extr: 20, nucl: 5, cyto: 5, lyso: 2",extr
7,g5705.t1,"extr: 29, plas: 1, mito: 1, lyso: 1",extr


,ID,Prediction,OTHER,SP,mTP,CS Position
17,g14932.t1,OTHER,0.999999,0.000001,0.000000,
15,g13005.t1,OTHER,0.999959,0.000007,0.000033,
2,g3481.t1,OTHER,0.999903,0.000033,0.000064,
8,g5712.t1,OTHER,0.999108,0.000016,0.000876,
11,g10874.t1,OTHER,0.998842,0.000266,0.000893,
9,g5811.t1,OTHER,0.997258,0.002727,0.000015,
12,g10875.t1,OTHER,0.996354,0.002113,0.001533,
6,g5703.t1,OTHER,0.994305,0.005695,0.000000,
14,g12948.t1,OTHER,0.994087,0.000064,0.005849,
13,g12947.t1,OTHER,0.992576,0.001304,0.006121,


,% Coverage,% Ident,Accession,Annotation,E-value,ID,Species
17,100.00,100.00,P0DOW4,RecName: Full=Damage suppressor protein [Ramaz...,0.000000e+00,g14932.t1,Ramazzottius varieornatus
2,91.81,56.60,Q09510,RecName: Full=Myosin regulatory light chain; A...,8.778600e-65,g3481.t1,Caenorhabditis elegans
3,74.04,27.70,O62243,RecName: Full=Zinc metalloproteinase nas-1; Al...,3.810480e-22,g3725.t1,Caenorhabditis elegans
15,53.56,38.60,Q6DRC4,RecName: Full=Eukaryotic translation initiatio...,6.124900e-52,g13005.t1,Danio rerio
18,46.89,39.76,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,1.891390e-14,g15627.t1,Ethmostigmus rubripes
1,44.04,37.21,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,1.790140e-12,g1408.t1,Ethmostigmus rubripes
0,39.07,40.48,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,1.419470e-11,g739.t1,Ethmostigmus rubripes
10,38.60,39.29,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,5.350550e-13,g5835.t1,Ethmostigmus rubripes
7,38.14,39.76,P0DPW4,RecName: Full=U-scoloptoxin(01)-Er1a; Short=U-...,6.747840e-14,g5705.t1,Ethmostigmus rubripes
4,NaN,NaN,None,None,NaN,g4156.t1,None


In [110]:
# 17k_model candidates:
blast_set = set(new_blast_df[new_blast_df['E-value'].isna() | (new_blast_df['E-value'] > 0.01)]['ID'])
targetp_set = set(new_targetp[new_targetp['Prediction'] == 'OTHER']['ID'])
wolf_set = set(new_wolf[new_wolf['Top Localization'] == 'nucl']['ID'])

print(blast_set & targetp_set & wolf_set)

{'g10874.t1', 'g10875.t1'}


In [112]:
my_sequence = get_sequence_by_id("diamond_my/candidate_proteins_python_my.faa", "g8803.t1")
pretrain_sequences = [get_sequence_by_id("diamond/candidate_proteins_python.faa", "g10513.t1"), \
    get_sequence_by_id("diamond/candidate_proteins_python.faa", "g10514.t1")]
new_sequences = [get_sequence_by_id("candidates_17k.fasta", "g10874.t1"), \
    get_sequence_by_id("candidates_17k.fasta", "g10875.t1")]

In [124]:
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

print("pretrain 1 (g10513.t1) vs new (g10874.t1)")
alignments_1 = pairwise2.align.globalxx(pretrain_sequences[0], new_sequences[0])
print(format_alignment(*alignments_1[0]))

# и со вторым
print("my (g8803.t1) vs new (g10875.t1)")
alignments_2 = pairwise2.align.globalxx(my_sequence, new_sequences[1])
print(format_alignment(*alignments_2[0]))

pretrain 1 (g10513.t1) vs new (g10874.t1)
M--------S-T-------T----------TSS--SS----------S--------NKDKDSTDTYVRSADNTSGSSNTTGSSSTNKGGKSSSSDYDSDKTTTSSSYGTGGQNTSSYGQSGQGGQHNTSTSSSYGQSGQGGQSGQHSSSSYGQSGQHSSGQSGQHSSGQSGQQSGQHGSHEVQQKLKEVGNLLQKAGHLLQDLQGSASDFSSSSSYSQRNQGGNYSGPYGGSDSQFHQYGSSGMGGGSYGQSSSYGQGSSGYGQGSSSYGQQSGRHQDSDRFGGSSSFGGQSSGQYGGHSQGGYGGQQGGYGGSSGQNYGPYXXXXXXXXXXXXX-------------RGMGGGYGFSQDSGRQGGMGMSGGMGGADRYGGFGGPNRPDMSYGQQYGGRSDRW
|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
MAPGNFPVDSSTALCLAFRTSRISLLPPSRTSSRVSSRFRLRLNITMSTTTSSSSSNKDKDSTDTYVRSADNTSGSSNTTGSSSTNKGGKSSSSDYDSDKTTTSSSYGTGGQNTSSYGQSGQGGQHNTSTSSSYGQSGQGGQSGQHSSSS

Наглядно видим превосходство глубокого секвенирования.

Модель, собранная исключительно на Sanger-данных, не нашла белок g10513.t1/g10874.t1, а модель с TSA-данными - нашла не только сам ген, но и дополнительный стартовый участок на 45 аминокислот.

g8803.t1 же уточнили, что он предсказан верно и стабильно на всех моделях.

Делаем вывод, что качество предсказания генов критически зависит от качества транскриптомных данных, а использование комбиннированных RNA-seq баз (Sanger + Illumina TSA) позволяет собрать полноразмерные белки.